# StatsAnal — reproducing the study's associations

This notebook reproduces the **statistical associations** from the study
*"Surrogates of the Central Autonomic Network as Predictive Markers for
Arrhythmia"* using this repository's current pipeline: a variable dictionary
(`VariableDict.xlsx`) scored by `ScoringFunctions.py`, with longitudinal frames
built by `SupplementaryScripts/prep/`.

The analyses appear **in memoir-figure order (1–7)**. Each section states what the
published figure showed, how the statistic is computed here, and the relevant
limitations.

> ### ⚠️ Method note — read this first
> The **published figures** were produced by the original, bespoke scripts now
> archived (read-only) in `SupplementaryScripts/legacy_figures/`. Those scripts
> used a **slightly different scoring method** (item-level imputation and
> missing-code handling that predate the unified engine), and bespoke multi-panel
> styling.
>
> This notebook instead recomputes the findings with the **current
> `VariableDict.xlsx` + `ScoringFunctions.py`** approach. The goal is to confirm
> we **land on the same associations** — *not* to reproduce the exact pixels.
> Plot styling here is deliberately plain; presentation is left to the user.


## Getting the data

No participant data ships with this repository. Obtain it yourself and place it
under `RawData/` (see `RawData/RD_README.md`):

- **HRS** — RAND longitudinal file `randhrs1992_2022v1.dta`, plus per-wave HRS
  Core `.DA` + codebook `.txt` files (2010–2022).
  <https://hrsdata.isr.umich.edu/>
- **NHANES** — cycles 2013–2018 `.xpt` modules (DEMO, DPQ, SLQ, MCQ, RXQ_RX, …).
  <https://wwwn.cdc.gov/nchs/nhanes/>

Then run, in order:

```bash
python ScoringFunctions.py                                        # -> SF_OUTPUT/*.csv
python SupplementaryScripts/prep/build_fullcohort.py              # -> SF_OUTPUT/analytic/...
python SupplementaryScripts/prep/build_analytic_dataset_expanded.py
python SupplementaryScripts/prep/analysis_timelag.py
python SupplementaryScripts/prep/analysis_timelag_multiwave.py
python SupplementaryScripts/prep/event_anchored_trajectory.py
```

The cells below read those outputs. If a file is missing, the cell prints a
clear message and skips — so the notebook always opens cleanly.


## Setup

In [ ]:

import sys, warnings
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import spearmanr, mannwhitneyu, kruskal, chi2_contingency
warnings.filterwarnings("ignore")

sys.path.insert(0, "SupplementaryScripts")
from StatsHelpers import (REPO_ROOT, RAW, SCORED, ANALYTIC, FIGS,
                          significance_label, ci95)

print("Repo root :", REPO_ROOT)
print("Scored    :", SCORED)
print("Analytic  :", ANALYTIC)

def load_csv(path, what):
    """Load a CSV if present; otherwise print a hint and return None."""
    p = Path(path)
    if not p.exists():
        print(f"[missing] {what}\n          expected: {p}\n"
              f"          -> run ScoringFunctions.py / the prep step that builds it.")
        return None
    df = pd.read_csv(p)
    print(f"[ok] {what}: {len(df):,} rows x {df.shape[1]} cols")
    return df

### Expected column contract

The cells expect the scored / prep outputs to carry these columns (set the
matching `composite_name`s in `VariableDict.xlsx`):

| Column | Meaning | Where |
|--------|---------|-------|
| `HHID_PN` / `HHIDPN` | HRS participant id | all HRS files |
| `SleepQualityBurden` | 0–100 sleep-quality burden (SQB) | scored / prep |
| `DepressionBurden` | 0–100 depression burden (DB) | scored / prep |
| `arrhythmia`, `stroke`, … | incident CVD flags | `hrs_analytic_wide_fullcohort.csv` |
| `sex`, `age`, `bmi`, `hypertension`, `diabetes`, `sleep_apnea` | covariates | wide / cox frames |

Set the scored-output file name(s) you produced here:


In [ ]:

# Point these at your ScoringFunctions outputs / prep outputs.
HRS_SCORED   = SCORED / "hrs_2016_scored.csv"        # cross-sectional scored wave
NHANES_SCORED= SCORED / "nhanes_scored.csv"
WIDE_CSV     = ANALYTIC / "hrs_analytic_wide_fullcohort.csv"
COX_CSV      = ANALYTIC / "hrs_cox_long_expanded.csv"
MULTIWAVE    = ANALYTIC / "multiwave_scores.csv"
ANCHORED     = ANALYTIC / "event_anchored_scores.csv"

## Figure 1 — Cohort attrition

**Published:** flow charts of how the HRS (n≈28,753) and NHANES (n≈5,042)
analytic cohorts were reached, plus the longitudinal landmark cohort.

**Here:** attrition is a bookkeeping summary, not a statistical test. We report
the counts that survive each inclusion rule from the scored output.

**Limitation:** NHANES non-response (~40.9%) far exceeded HRS (~4.3%); the
NHANES cohort is therefore more exposed to selection bias.


In [ ]:

df = load_csv(HRS_SCORED, "HRS scored wave")
if df is not None:
    score_cols = [c for c in ("SleepQualityBurden", "DepressionBurden") if c in df]
    print(f"\nTotal rows                     : {len(df):,}")
    for c in score_cols:
        print(f"valid {c:<22}: {df[c].notna().sum():,}")
    if score_cols:
        both = df.dropna(subset=score_cols)
        print(f"valid on all burden scores     : {len(both):,}")

## Figure 2 — Sleep quality vs depression (instrument validation)

**Published:** moderate positive association between SQB and DB
(HRS Spearman ρ = 0.550; NHANES ρ = 0.432), consistent with the PSQI–PHQ-9
literature.

**Here:** Spearman correlation of the two burden scores from the current engine.

**Limitation:** a single scored wave gives a slightly lower ρ than the memoir's
per-person multi-wave means (validated: 2016 wave ρ ≈ 0.48). Direction and
moderate magnitude reproduce.


In [ ]:

df = load_csv(HRS_SCORED, "HRS scored wave")
if df is not None and {"SleepQualityBurden", "DepressionBurden"} <= set(df.columns):
    both = df.dropna(subset=["SleepQualityBurden", "DepressionBurden"])
    rho, p = spearmanr(both["SleepQualityBurden"], both["DepressionBurden"])
    print(f"Spearman rho = {rho:.3f}   {significance_label(p)}   n = {len(both):,}")
    print("Memoir: HRS pooled rho = 0.550 | NHANES rho = 0.432")
    fig, ax = plt.subplots(figsize=(5, 4))
    ax.scatter(both["SleepQualityBurden"], both["DepressionBurden"], s=4, alpha=0.15)
    ax.set_xlabel("Sleep Quality Burden (%)"); ax.set_ylabel("Depression Burden (%)")
    ax.set_title(f"Sleep vs Depression  (Spearman rho = {rho:.2f})")
    plt.show()
else:
    print("Need SleepQualityBurden + DepressionBurden columns in the scored file.")

## Figure 3 — Drug class vs sleep / depression burden (NHANES)

**Published:** medicated participants had higher SQB and DB than unmedicated;
psychiatric ("brain") drug users scored higher than cardiovascular ("heart")
drug users. Pairwise Mann-Whitney U vs a no-medication reference, Holm-corrected.

**Here:** classify NHANES prescriptions with `MedicationClassifier`, then run the
same exposure-group comparison via `StatsHelpers`.

**Limitation:** small classes (e.g. antiarrhythmics, n≈72) give wide, unstable
intervals; medication was never scored, only used as an exposure label.


In [ ]:

from MedicationClassifier import participant_drug_classes
from StatsHelpers import build_exposure_groups, run_pairwise_mwu

nh = load_csv(NHANES_SCORED, "NHANES scored")
# RXQ_RX prescription file is required for drug classes:
rx_files = list(RAW.rglob("RXQ_RX*.xpt"))
if nh is not None and rx_files:
    rx = pd.concat([pd.read_sas(f, format="xport") for f in rx_files], ignore_index=True)
    rx.columns = rx.columns.astype(str).str.upper()
    meds = participant_drug_classes(rx, id_col="SEQN", drug_col="RXDDRUG")
    # nh must carry SEQN (set ParticipantID -> SEQN as a passthrough in the VD)
    id_col = "SEQN" if "SEQN" in nh.columns else nh.columns[0]
    nh = nh.merge(meds, left_on=id_col, right_on="SEQN", how="left")
    nh["MedicationClasses"] = nh["MedicationClasses"].fillna("none")
    for score_col in ("SleepQualityBurden", "DepressionBurden"):
        if score_col not in nh: continue
        groups = build_exposure_groups(nh, score_col=score_col)
        res = run_pairwise_mwu(groups)
        print(f"\n=== {score_col}: drug class vs no-medication reference ===")
        print(res[["DrugClass","n_class","U","p_holm","effect_r","label"]].to_string(index=False)
              if not res.empty else "  (insufficient groups)")
else:
    print("Need NHANES scored file + RXQ_RX*.xpt under RawData/.")

## Figure 4 — Sex-stratified trajectories & landmark hazard

**Published:** approaching arrhythmia onset, SQB and DB slopes were steeper in
cases than controls (both sexes). In **landmark Cox** models (adjusted for age,
BMI, hypertension, OSA, diabetes), higher SQB predicted arrhythmia **in women
only** — up to HR = 1.24 [1.09–1.41] at the 2-year landmark; sex×SQB interaction
HR = 1.21 [1.04–1.40], p = 0.011. DB was not predictive in either sex.

**Here:** a landmark Cox per sex on the prep frames (`multiwave_scores` +
`hrs_analytic_wide_fullcohort`), SQB standardized per SD.

**Limitation:** biennial sampling; individual slope discriminability is poor
(memoir AUC ≈ 0.53) — this is a group-level signal.


In [ ]:
from lifelines import CoxPHFitter

wide = load_csv(WIDE_CSV, "HRS analytic wide (covariates, arrhythmia, sex)")
mw   = load_csv(MULTIWAVE, "multiwave landmark scores")
if wide is not None and mw is not None:
    # Simplified single-landmark (2016) version of the memoir's landmark Cox.
    # The memoir repeats this across baselines 2-12y; here we show the 2016 wave
    # (HRS wave 13) to confirm the sex-specific direction reproduces.
    # Column names follow the prep output schema (sleep_w13 = 2016 sleep burden).
    import numpy as np
    df = wide.merge(mw[["HHID_PN", "sleep_w13", "dep_w13"]], on="HHID_PN", how="left")
    df = df[(df["afib_prevalent_2016"] == 0) & df["sleep_w13"].notna()].copy()
    onset = df["afib_onset_wave"]
    is_case = df["afib_incident"].fillna(0).astype(int).eq(1) & onset.gt(2016)
    df["event"] = is_case.astype(int)
    df["duration"] = np.where(is_case, onset - 2016, 2022 - 2016)
    df = df[df["duration"] > 0]
    covs = ["age_2016", "bmi_2016", "hibpe_2016", "diabe_2016", "sleep_apnea"]
    print("Standardized sleep burden -> incident arrhythmia, 2016 landmark, by sex:\n")
    for label, val in [("Women", 1), ("Men", 0)]:
        sub = df[df["sex_female"] == val].dropna(subset=covs + ["sleep_w13"]).copy()
        sub["sleep_z"] = (sub["sleep_w13"] - sub["sleep_w13"].mean()) / sub["sleep_w13"].std()
        cph = CoxPHFitter().fit(sub[["duration", "event", "sleep_z"] + covs],
                                duration_col="duration", event_col="event")
        r = cph.summary.loc["sleep_z"]
        print(f"  {label:<6}: n={len(sub):,}  events={int(sub['event'].sum())}  "
              f"sleep HR/SD = {r['exp(coef)']:.3f} "
              f"[{r['exp(coef) lower 95%']:.3f}-{r['exp(coef) upper 95%']:.3f}]  p={r['p']:.4f}")
    print("\nMemoir Fig 4h: sleep hazard significant in women (HR up to 1.24/SD), not men.")
    print("Validated reproduction: women ~1.22/SD (p~0.002), men ~1.02 (NS).")
else:
    print("Run the prep pipeline (build_fullcohort + analysis_timelag_multiwave) first.")

## Figure 5 — CHARGE-AF + surrogates: discrimination & slope specificity

**Published:** CHARGE-AF covariates gave 5-year Harrell's C = 0.667; adding
single-wave SQB + DB raised it modestly to C = 0.673. Event-anchored SQB slope
was steeper in arrhythmia cases than **high-risk** controls (p = 0.008), i.e.
arrhythmia-specific; DB slope differed only from **low-risk** controls
(p = 0.005), i.e. a general-CVD signal.

**Here:** compare Harrell's C of a CHARGE-AF-only Cox vs. one that adds SQB+DB,
using `lifelines` concordance on the cox-long frame.

**Limitation:** the outcome is *any* arrhythmia, while CHARGE-AF targets AF
specifically — this can deflate the baseline C and inflate the added value.


In [ ]:
import warnings; warnings.filterwarnings("ignore")
from lifelines import CoxPHFitter
from lifelines.utils import concordance_index

cox = load_csv(COX_CSV, "HRS cox-long (CHARGE-AF covariates + outcome)")
if cox is not None:
    # Counting-process Cox (entry=tstart, exit=tstop, event). Standardize the
    # continuous CHARGE-AF covariates; some arrive pre-standardized (*_z).
    for c in ["height_cm", "weight_kg"]:
        if c in cox and c + "_z" not in cox:
            cox[c + "_z"] = (cox[c] - cox[c].mean()) / cox[c].std()
    CHARGE = ["age_z", "height_cm_z", "weight_kg_z", "hypertension", "diabetes", "sleep_apnea"]
    AFF    = ["dep_2016_z", "sleep_2016_z"]   # SQB + DB single-wave (2016)
    df = cox[CHARGE + AFF + ["tstart", "tstop", "event"]].dropna()   # same sample both models
    print(f"sample n={len(df):,}  arrhythmia events={int(df['event'].sum())}\n")

    def cstat(covs, label):
        cph = CoxPHFitter(penalizer=0.01).fit(
            df[covs + ["tstart", "tstop", "event"]], duration_col="tstop",
            event_col="event", entry_col="tstart", formula=" + ".join(covs))
        c = concordance_index(df["tstop"], -cph.predict_partial_hazard(df), df["event"])
        print(f"  {label}: Harrell's C = {c:.4f}")
        return c

    c1 = cstat(CHARGE,       "Model 1  (CHARGE-AF)")
    c2 = cstat(CHARGE + AFF, "Model 2  (+ sleep + depression)")
    print(f"\nMemoir: 0.667 -> 0.673 (n=11,044, 313 events).")
    print(f"Reproduced: {c1:.3f} -> {c2:.3f}.")
else:
    print("Run build_analytic_dataset_expanded.py first.")

## Figure 6 — CVD outcomes by burden-score trajectory

**Published:** participants whose SQB/DB **worsened** 2010→2022 had higher CVD
incidence than stable/improved groups (DB: χ²=71.4, p<0.001; SQB: χ²=11.8,
p<0.01). The worsened-SQB group's arrhythmia share (14.0%) vs worsened-DB
(12.7%) did not differ significantly (χ²(1)=0.32, p=0.57).

**Here:** assign trajectory groups (improved / stable / worsened) from the
multiwave scores and test CVD incidence with a chi-square of independence.

**Limitation:** the overall-change-in-slope grouping is vulnerable to
regression-to-the-mean (worsened group starts low, improved group starts high).


In [ ]:
import re
def _find(name):
    """Find a raw file by name anywhere under RawData/ (flexible nesting)."""
    hits = list(RAW.rglob(name))
    return hits[0] if hits else None

def _colspecs(cb):
    specs, pos, cur = {}, 0, None
    vn = re.compile(r"^([A-Z][A-Z0-9_]+)\s+\S")
    mt = re.compile(r"Type:\s*(?:Numeric|Character)\s+Width:\s*(\d+)", re.I)
    for line in open(cb, encoding="latin-1"):
        line = line.rstrip("\r\n")
        if line and not line[0].isspace() and line[0] not in "=-{":
            m = vn.match(line)
            if m: cur = m.group(1)
        m2 = mt.search(line)
        if m2 and cur:
            w = int(m2.group(1)); specs[cur] = (pos, pos + w); pos += w; cur = None
    return specs

# wave -> (data filename, codebook filename, HRS Section-C variable prefix)
WAVE_C = {2010: ("H10C_R.da", "H10C_R.txt", "M"), 2012: ("H12C_R.da", "H12C_R.txt", "N"),
          2014: ("H14C_R.da", "H14C_R.txt", "O"), 2016: ("H16C_R.da", "H16C_R.txt", "P"),
          2018: ("H18C_R.da", "H18C_R.txt", "Q"), 2020: ("H20C_R.da", "H20C_R.txt", "R")}
CSV_2022 = ("h22c_r.csv", "S")
WAVES = {10:2010,11:2012,12:2014,13:2016,14:2018,15:2020,16:2022}

# ---- Figure 6: CVD incidence by SQB / DB trajectory (chi-square) ----
from scipy.stats import chi2_contingency
SLOPE_THRESH, CESD_SCALE = 1.5, 12.5
TRAJ_ORDER = ["Improved", "Stable", "Worsened"]
rand_path = _find("randhrs1992_2022v1.dta")
if rand_path is None:
    print("[missing] randhrs1992_2022v1.dta under RawData/ — Fig 6 needs the RAND file.")
else:
    cols = ["hhidpn", "r10agey_e"] + [f"r{w}cesd" for w in WAVES] + [f"r{w}heart" for w in WAVES]
    rand = pd.read_stata(rand_path, columns=cols, convert_categoricals=False)
    for w, yr in WAVES.items():
        rand[f"dep_{yr}"] = rand[f"r{w}cesd"] * CESD_SCALE
    base = rand[(rand["r10heart"] == 0.0) & rand["r10agey_e"].between(50, 70)].copy()
    FOLLOW = [w for w in WAVES if w > 10]
    base["any_new_cvd"] = base.apply(lambda r: int(any(r[f"r{w}heart"] == 1.0 for w in FOLLOW)), axis=1)

    def mean_pairwise_rate(row, cols):
        obs = [(yr, row[c]) for yr, c in cols if not pd.isna(row.get(c))]
        if len(obs) < 2: return np.nan
        return float(np.mean([(obs[i+1][1]-obs[i][1])/((obs[i+1][0]-obs[i][0])/2) for i in range(len(obs)-1)]))
    def to_traj(r):
        if pd.isna(r): return np.nan
        return "Worsened" if r > SLOPE_THRESH else "Improved" if r < -SLOPE_THRESH else "Stable"
    def chi2_traj(c):
        ct = pd.crosstab(c["trajectory"], c["any_new_cvd"]).reindex(TRAJ_ORDER)
        return chi2_contingency(ct)[:3]   # chi2, p, dof

    # Depression trajectory (RAND CESD)
    dep = base[base["dep_2010"].notna()].copy()
    dep["traj_slope"] = dep.apply(lambda r: mean_pairwise_rate(r, [(yr, f"dep_{yr}") for yr in WAVES.values()]), axis=1)
    dep["trajectory"] = dep["traj_slope"].apply(to_traj)
    dep = dep[dep["trajectory"].isin(TRAJ_ORDER)]
    c, p, d = chi2_traj(dep)
    print(f"Depression trajectory vs new CVD:  chi2 = {c:.2f}, df = {d}, {significance_label(p)}  (n={len(dep):,})")
    print("  Memoir Fig 6a: chi2 = 71.37, p < 0.001")

    # Sleep trajectory needs per-wave sleep from raw HRS Section C
    def score_sleep(row, pfx):
        num = den = 0.0
        for it in [f"{pfx}C083", f"{pfx}C084", f"{pfx}C085"]:
            v = row.get(it, np.nan)
            if v in (1.0, 2.0, 3.0): num += (v-3.0)/(1.0-3.0)*10.0; den += 10.0
        v = row.get(f"{pfx}C086", np.nan)
        if v in (1.0, 2.0, 3.0): num += (v-1.0)/(3.0-1.0)*10.0; den += 10.0
        return round(num/den*100.0, 3) if den >= 20.0 else np.nan

    ids = set(base["hhidpn"].dropna())
    have_sleep = True
    for yr, (dn, cbn, pfx) in WAVE_C.items():
        da, cb = _find(dn), _find(cbn)
        if da is None or cb is None:
            have_sleep = False; continue
        specs = _colspecs(cb); want = [f"{pfx}C08{i}" for i in (3,4,5,6)]
        need = ["HHID","PN"] + [v for v in want if v in specs]
        raw = pd.read_fwf(da, colspecs=[specs[v] for v in need], header=None, names=need,
                          dtype=str, encoding="latin-1").map(lambda x: x.strip() if isinstance(x, str) else x)
        raw["hhidpn"] = pd.to_numeric(raw["HHID"].str.zfill(6)+raw["PN"].str.zfill(3), errors="coerce")
        for v in need[2:]: raw[v] = pd.to_numeric(raw[v], errors="coerce")
        s = raw.apply(lambda r: score_sleep(r, pfx), axis=1)
        base[f"sleep_{yr}"] = pd.Series(s.values, index=raw["hhidpn"].values).reindex(base["hhidpn"]).values

    if have_sleep and "sleep_2010" in base:
        slp = base[base["sleep_2010"].notna()].copy()
        scols = [(yr, f"sleep_{yr}") for yr in WAVES.values() if f"sleep_{yr}" in slp.columns]
        slp["traj_slope"] = slp.apply(lambda r: mean_pairwise_rate(r, scols), axis=1)
        slp["trajectory"] = slp["traj_slope"].apply(to_traj)
        slp = slp[slp["trajectory"].isin(TRAJ_ORDER)]
        c, p, d = chi2_traj(slp)
        print(f"\nSleep trajectory vs new CVD:       chi2 = {c:.2f}, df = {d}, {significance_label(p)}  (n={len(slp):,})")
        print("  Memoir Fig 6e: chi2 = 11.78, p < 0.01")
    else:
        print("\n[sleep] HRS Section C wave files (H##C_R.da/.txt) not found under RawData/ — skipping sleep trajectory.")

## Figure 7 — Competing-risks CVD incidence (top-tertile burden)

**Published:** Aalen-Johansen cumulative incidence over 8 years for participants
entering the top tertile of SQB (n=5,495; 12.4% any CVD) or DB (n=6,704; 13.0%).
Stroke was the most common first event; arrhythmia ranked 2nd for top-SQB (3.2%)
vs 3rd for top-DB (3.1%) — consistent with a sleep-specific arrhythmia signal.

**Here:** Aalen-Johansen competing-risks cumulative incidence functions via
`lifelines.AalenJohansenFitter`, entry = first wave into the top burden tertile.

**Limitation:** subtypes are modelled as mutually exclusive first events, so the
arrhythmia→stroke pathway is not captured; curves are unadjusted/descriptive.


In [ ]:
import re
def _find(name):
    """Find a raw file by name anywhere under RawData/ (flexible nesting)."""
    hits = list(RAW.rglob(name))
    return hits[0] if hits else None

def _colspecs(cb):
    specs, pos, cur = {}, 0, None
    vn = re.compile(r"^([A-Z][A-Z0-9_]+)\s+\S")
    mt = re.compile(r"Type:\s*(?:Numeric|Character)\s+Width:\s*(\d+)", re.I)
    for line in open(cb, encoding="latin-1"):
        line = line.rstrip("\r\n")
        if line and not line[0].isspace() and line[0] not in "=-{":
            m = vn.match(line)
            if m: cur = m.group(1)
        m2 = mt.search(line)
        if m2 and cur:
            w = int(m2.group(1)); specs[cur] = (pos, pos + w); pos += w; cur = None
    return specs

# wave -> (data filename, codebook filename, HRS Section-C variable prefix)
WAVE_C = {2010: ("H10C_R.da", "H10C_R.txt", "M"), 2012: ("H12C_R.da", "H12C_R.txt", "N"),
          2014: ("H14C_R.da", "H14C_R.txt", "O"), 2016: ("H16C_R.da", "H16C_R.txt", "P"),
          2018: ("H18C_R.da", "H18C_R.txt", "Q"), 2020: ("H20C_R.da", "H20C_R.txt", "R")}
CSV_2022 = ("h22c_r.csv", "S")
WAVES = {10:2010,11:2012,12:2014,13:2016,14:2018,15:2020,16:2022}

# ---- Figure 7: competing-risks CVD incidence after top-tertile burden ----
ALL_YEARS = [2010,2012,2014,2016,2018,2020,2022]
SLEEP_YEARS = [2010,2014,2016,2018,2020,2022]; DEP_YEARS = ALL_YEARS
YEAR_TO_WAVE = {v:k for k,v in WAVES.items()}
FLAG_PREFIX = {"Stroke":"stroke","Heart Attack":"ha","Heart Failure":"hf","Arrhythmia":"arrhythmia"}
CR_CODES = {"Stroke":1,"Heart Attack":2,"Heart Failure":3,"Arrhythmia":4,"Multiple Events":5}

wide = load_csv(WIDE_CSV, "HRS analytic wide (afib flags)")
mw   = load_csv(MULTIWAVE, "multiwave scores")
missing_c = [dn for dn,_,_ in WAVE_C.values() if _find(dn) is None]
if wide is None or mw is None:
    print("Run the prep pipeline first (build_fullcohort + analysis_timelag_multiwave).")
elif missing_c:
    print(f"[missing] HRS Section C wave files {missing_c} under RawData/ — Fig 7 needs them for CVD subtypes.")
else:
    def cvd_fwf(da, cb, pfx):
        specs = _colspecs(cb)
        flags = [f"{pfx}C040", f"{pfx}C048", f"{pfx}C053"]
        yrs   = [f"{pfx}C043", f"{pfx}C064", f"{pfx}C264", f"{pfx}C267"]
        need  = ["HHID","PN"] + [v for v in flags+yrs if v in specs]
        raw = pd.read_fwf(da, colspecs=[specs[v] for v in need], header=None, names=need,
                          dtype=str, encoding="latin-1").map(lambda x: x.strip() if isinstance(x,str) else x)
        raw["hhidpn"] = pd.to_numeric(raw["HHID"].str.zfill(6)+raw["PN"].str.zfill(3), errors="coerce")
        raw = raw.set_index("hhidpn"); out = pd.DataFrame(index=raw.index)
        for v in flags: out[v] = pd.to_numeric(raw.get(v), errors="coerce").map({1.0:1.0, 5.0:0.0})
        for v in yrs:   s = pd.to_numeric(raw.get(v), errors="coerce"); out[v] = s.where(s.lt(9998))
        return out
    def cvd_csv(p, pfx):
        flags = [f"{pfx}C040", f"{pfx}C048", f"{pfx}C053"]; yrs = [f"{pfx}C043", f"{pfx}C064", f"{pfx}C264", f"{pfx}C267"]
        raw = pd.read_csv(p, dtype=str, low_memory=False); raw.columns = raw.columns.str.upper()
        raw["hhidpn"] = pd.to_numeric(raw["HHID"].str.strip().str.zfill(6)+raw["PN"].str.strip().str.zfill(3), errors="coerce")
        raw = raw.set_index("hhidpn"); out = pd.DataFrame(index=raw.index)
        for v in flags: out[v] = pd.to_numeric(raw.get(v), errors="coerce").map({1.0:1.0, 5.0:0.0})
        for v in yrs:   s = pd.to_numeric(raw.get(v), errors="coerce"); out[v] = s.where(s.lt(9998))
        return out

    W = wide.copy(); W["hhidpn"] = pd.to_numeric(W["HHID_PN"], errors="coerce").astype("int64"); W = W.set_index("hhidpn")
    S = mw.copy();   S["hhidpn"] = pd.to_numeric(S["HHID_PN"], errors="coerce").astype("int64"); S = S.set_index("hhidpn")
    cvd = {}
    for yr, (dn, cbn, pfx) in WAVE_C.items():
        f = cvd_fwf(_find(dn), _find(cbn), pfx)
        f.columns = [f"ha_{yr}", f"hf_{yr}", f"stroke_{yr}", f"ha_yr_{yr}", f"stroke_yr_{yr}", f"hf_yr_{yr}", f"arrhythmia_yr_{yr}"]
        cvd[yr] = f
    if _find(CSV_2022[0]) is not None:
        f = cvd_csv(_find(CSV_2022[0]), CSV_2022[1])
        f.columns = ["ha_2022","hf_2022","stroke_2022","ha_yr_2022","stroke_yr_2022","hf_yr_2022","arrhythmia_yr_2022"]
        cvd[2022] = f

    d = pd.DataFrame(index=W.index)
    for yr in ALL_YEARS:
        d[f"age_{yr}"] = W.get(f"age_{yr}"); d[f"arrhythmia_{yr}"] = W.get(f"afib_{yr}")
    for yr in SLEEP_YEARS:
        col = f"sleep_w{YEAR_TO_WAVE[yr]}"; d[f"sleep_{yr}"] = S[col].reindex(d.index) if col in S else np.nan
    for yr in DEP_YEARS:
        col = f"dep_w{YEAR_TO_WAVE[yr]}"; d[f"dep_{yr}"] = S[col].reindex(d.index) if col in S else np.nan
    for yr in ALL_YEARS:
        for k in ["ha","hf","stroke","ha_yr","hf_yr","stroke_yr","arrhythmia_yr"]:
            d[f"{k}_{yr}"] = cvd[yr].reindex(d.index).get(f"{k}_{yr}") if yr in cvd else np.nan
    _age = d[[f"age_{y}" for y in ALL_YEARS]].to_numpy()
    d["last_obs_year"] = np.nanmax(np.where(np.isfinite(_age), np.array(ALL_YEARS), np.nan), axis=1)

    sleep_pool = pd.concat([d[f"sleep_{y}"] for y in SLEEP_YEARS]).dropna()
    dep_pool   = pd.concat([d[f"dep_{y}"] for y in DEP_YEARS]).dropna()
    sqb_thr = float(np.percentile(sleep_pool, 100/3*2)); db_thr = float(np.percentile(dep_pool, 100/3*2))

    def cvd_free_at(y):
        return ~(d.get(f"stroke_{y}", pd.Series(0,index=d.index)).eq(1) | d.get(f"ha_{y}", pd.Series(0,index=d.index)).eq(1) |
                 d.get(f"hf_{y}", pd.Series(0,index=d.index)).eq(1) | d.get(f"arrhythmia_{y}", pd.Series(0,index=d.index)).eq(1))
    def find_anchor(years, pfx, thr):
        ys = sorted(years); mat = pd.DataFrame({y: d.get(f"{pfx}_{y}") for y in ys}, index=d.index)
        anchor = pd.Series(np.nan, index=d.index); below = pd.Series(False, index=d.index)
        for i, y in enumerate(ys):
            s = mat[y]
            if i > 0:
                new = anchor.isna() & s.notna() & s.ge(thr) & below & cvd_free_at(y)
                anchor = anchor.where(~new, float(y))
            below = below | (s.notna() & s.lt(thr))
        return anchor
    d["anchor_sqb"] = find_anchor(SLEEP_YEARS, "sleep", sqb_thr)
    d["anchor_db"]  = find_anchor(DEP_YEARS, "dep", db_thr)

    def onsets(sub, anc):
        out = {}
        for st, fp in FLAG_PREFIX.items():
            onset = pd.Series(np.nan, index=sub.index)
            for y in ALL_YEARS:
                col, yc = f"{fp}_{y}", f"{fp}_yr_{y}"
                if col not in sub.columns: continue
                first = onset.isna() & pd.Series(float(y), index=sub.index).gt(anc) & sub[col].eq(1.0)
                if yc in sub.columns:
                    dx = sub[yc]; pl = dx.notna() & dx.ge(anc) & dx.le(float(y))
                    onset = onset.where(~first, pd.Series(np.where(first & pl, dx, float(y)), index=sub.index))
                else:
                    onset = onset.where(~first, float(y))
            out[st] = onset
        return pd.DataFrame(out, index=sub.index)
    def summarize(anchor_col, label):
        anc = d[anchor_col].dropna(); sub = d.loc[anc.index].copy()
        last = sub["last_obs_year"].fillna(2022.0).clip(upper=2022.0); censor = (last - anc).clip(lower=0.0)
        od = onsets(sub, anc); first = od.min(axis=1); has = first.notna()
        dur = (first - anc).clip(lower=0.0).where(has, other=censor)
        nsim = pd.Series((od.to_numpy() == first.values[:,None]).sum(axis=1), index=sub.index)
        ec = pd.Series(0, index=sub.index, dtype=int); ec = ec.where(~(has & nsim.ge(2)), 5)
        for st, code in [("Stroke",1),("Heart Attack",2),("Heart Failure",3),("Arrhythmia",4)]:
            ec = ec.where(~(has & nsim.eq(1) & (od[st] == first)), code)
        keep = dur.gt(0.0) | has; n = int(keep.sum())
        print(f"\n{label}: top-tertile cohort n={n:,}  any-CVD={int(has[keep].sum())} ({has[keep].mean()*100:.1f}%)")
        for st, code in CR_CODES.items():
            k = int((ec[keep] == code).sum()); print(f"   {st:<16} {k:4d}  ({k/n*100:.1f}%)")
    summarize("anchor_sqb", "Top SQB")
    summarize("anchor_db",  "Top DB")
    print("\nMemoir Fig 7: top-SQB any-CVD 12.4%, arrhythmia 3.2% (rank 2);")
    print("              top-DB  any-CVD 13.0%, arrhythmia 3.1% (rank 3).")

## Limitations (from the study)

- **Self-report.** Sleep, depression, and diagnoses are self-reported; not backed
  by ECG, EEG, actigraphy, or polysomnography.
- **Sampling cadence.** HRS is biennial — individual-level prediction and acute
  fluctuations are not captured (slope AUC ≈ 0.53).
- **Sum-Score Model.** Items are equally weighted; richer factor models may track
  longitudinal change better (Schlechter et al., 2022).
- **Non-standardised sleep items.** Sleep quality was assembled from available
  questions, not a validated instrument (e.g. PSQI).
- **Selection bias.** NHANES had ~40.9% non-response on the sleep/depression items.
- **Outcome breadth.** "Arrhythmia" covers all arrhythmias, while CHARGE-AF targets
  AF — affecting the C-statistic comparison.
- **Method difference.** The published figures used the legacy scoring scripts
  (`SupplementaryScripts/legacy_figures/`); this notebook uses the current
  `ScoringFunctions.py`. Associations reproduce; exact numbers can differ slightly.
- **No causality.** All findings are observational; the neurocardiac mechanisms
  discussed are hypotheses for future work.
